# 05 — Fine-Tuned Model Evaluation

Phase 10. Evaluates on **exactly the same images** and **exactly the same
prompts** as the baseline. Conditions are not adjusted to flatter the
fine-tuned model.


In [ ]:
# --- Colab setup (skip if running locally) ---
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('casting-defect-vlm'):
        # Replace with your repository URL, or upload the folder to Colab.
        raise SystemExit('Upload the casting-defect-vlm project folder to Colab first.')
    %cd casting-defect-vlm
    !pip install -q -r requirements.txt

sys.path.insert(0, os.path.abspath('..' if os.path.basename(os.getcwd())=='notebooks' else '.'))
print('python', sys.version.split()[0], '| colab:', IN_COLAB)


In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('WARNING: no CUDA GPU. Baseline and training need one.')


## 1. Confirm the frozen subset


In [ ]:
import json
from pathlib import Path
sub = json.loads(Path('results/baseline/eval_subset.json').read_text())
print('frozen subset size:', sub['n'])
print('class distribution:', sub['class_distribution'])
print(sub['note'])


## 2. Run evaluation


In [ ]:
!python scripts/evaluate_model.py


## 3. Metrics


In [ ]:
import pandas as pd
m = json.loads(Path('results/evaluation/finetuned_metrics.json').read_text())['overall']
for k in ['accuracy','precision_defective','recall_defective','f1_defective',
          'macro_f1','n_unparseable']:
    print(f'{k:<24}{m.get(k)}')
print(); print(m['classification_report_text'])


## 4. Confusion matrix


In [ ]:
from IPython.display import Image, display
display(Image('results/evaluation/finetuned_confusion_matrix.png'))


## 5. Raw outputs


In [ ]:
with open('results/evaluation/finetuned_raw_outputs.jsonl') as fh:
    raws = [json.loads(l) for l in fh]
for r in raws[:5]:
    print('='*70)
    print(r['image_id'], '| truth:', r['ground_truth'], '| prompt:', r['prompt_id'])
    print(r['raw_response'])
    print('->', r['parsed_prediction'], '| correct:', r['correct'])


## 6. Error analysis


In [ ]:
errs = pd.read_csv('results/evaluation/error_analysis.csv')
print(errs['error_type'].value_counts())
errs.head(12)


## 7. Defect-type behaviour

**Caveat:** these defect types are synthetic. Any accuracy here describes
recognition of generated defect textures, not real casting defects.


In [ ]:
res = pd.read_csv('results/evaluation/finetuned_results.csv')
d = res[res['ground_truth']=='Defective']
print(pd.crosstab(d['true_defect_type'], d['predicted_defect_type']).head(15))
